# Uganda Crime Hotspot Prediction + GIS Heatmap + Streamlit Dashboard

Full Colab notebook: EDA, ML models, threshold tuning, hotspot analysis, static Uganda map, Folium heatmap, and a Streamlit interface launched from Colab.

Dataset: `/content/drive/MyDrive/Group 12 Project/Uganda_Crime_Dataset/uganda_crime_ml_expanded.csv`
Outputs: `/content/drive/MyDrive/FinalYearProject`

In [1]:
!pip -q install xgboost folium streamlit
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 55.1 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [2]:
import os, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE=True
except Exception:
    XGB_AVAILABLE=False
import folium
from folium.plugins import HeatMap
pd.set_option('display.max_columns', 200)

In [3]:
import sklearn
print(sklearn.__version__)

1.6.1


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
DATA_PATH = '/content/drive/MyDrive/Group 12 Project/Uganda_Crime_Dataset/uganda_crime_ml_expanded.csv'
BASE_DIR = Path('/content/drive/MyDrive/Group 12 Project')
FIG_DIR, TABLE_DIR, MAP_DIR, APP_DIR = BASE_DIR/'figures', BASE_DIR/'tables', BASE_DIR/'maps', BASE_DIR/'streamlit_app'
for d in [FIG_DIR, TABLE_DIR, MAP_DIR, APP_DIR]: d.mkdir(parents=True, exist_ok=True)
print('Saving outputs to:', BASE_DIR)

Saving outputs to: /content/drive/MyDrive/Group 12 Project


In [7]:
df = pd.read_csv(DATA_PATH)
df.columns = [str(c).strip().lower().replace(' ','_').replace('/','_').replace('-','_') for c in df.columns]
for col in df.select_dtypes(include='object').columns: df[col]=df[col].fillna('Unknown').astype(str).str.strip()
for col in df.select_dtypes(include=[np.number]).columns: df[col]=df[col].fillna(df[col].median())
df['high_crime_region'] = (df['total_cases'] > df['total_cases'].quantile(0.80)).astype(int)
print('Shape:', df.shape)
print(df['high_crime_region'].value_counts())
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Group 12 Project/Uganda_Crime_Dataset/uganda_crime_ml_expanded.csv'

In [ ]:
summary_table = df.describe(include='all').T
display(summary_table)
summary_table.to_csv(TABLE_DIR/'dataset_summary.csv')

In [ ]:
plt.figure(figsize=(6,4))
df['high_crime_region'].value_counts().sort_index().plot(kind='bar')
plt.title('Target Distribution')
plt.xlabel('Class'); plt.ylabel('Count'); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(FIG_DIR/'target_distribution.png', dpi=300); plt.show()

In [ ]:
year_summary = df.groupby('year')['total_cases'].mean().sort_index()
display(year_summary)
plt.figure(figsize=(8,5)); year_summary.plot(marker='o')
plt.title('Average Total Cases by Year'); plt.xlabel('Year'); plt.ylabel('Average Total Cases'); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR/'average_cases_by_year.png', dpi=300); plt.show()

In [ ]:
region_summary = df.groupby('region')['total_cases'].mean().sort_values(ascending=False)
display(region_summary.head(20))
plt.figure(figsize=(12,6)); region_summary.plot(kind='bar')
plt.title('Average Total Cases by Region'); plt.xlabel('Region'); plt.ylabel('Average Total Cases'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig(FIG_DIR/'average_cases_by_region.png', dpi=300); plt.show()

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()
display(corr)
plt.figure(figsize=(9,7)); plt.imshow(corr, aspect='auto'); plt.colorbar(); plt.title('Correlation Matrix')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90); plt.yticks(range(len(corr.columns)), corr.columns)
plt.tight_layout(); plt.savefig(FIG_DIR/'correlation_matrix.png', dpi=300); plt.show()
corr.to_csv(TABLE_DIR/'correlation_matrix.csv')

In [ ]:
drop_cols = ['high_crime_region','total_cases','case_change_vs_prev_year','pct_change_vs_prev_year','region_case_rank_in_year']
X = df.drop(columns=drop_cols, errors='ignore').copy()
y = df['high_crime_region'].copy()
if 'year' in X.columns:
    np.random.seed(42); X['year_noisy'] = X['year'] + np.random.normal(0,0.5,size=len(X)); X = X.drop(columns=['year'], errors='ignore')
if 'region' in X.columns:
    freq = df['region'].value_counts(normalize=True); X['region_freq'] = df['region'].map(freq); X = X.drop(columns=['region'])
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
print('Features used:', X.columns.tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
preprocessor = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features)], remainder='drop')
print(X_train.shape, X_test.shape)

In [ ]:
models = {
 'Logistic Regression': LogisticRegression(max_iter=3000, class_weight={0:1,1:2}, random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=15, min_samples_leaf=15, class_weight={0:1,1:2}, random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_split=10, min_samples_leaf=5, class_weight={0:1,1:2}, random_state=42, n_jobs=-1),
 'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=.05, max_depth=3, random_state=42),
 'Extra Trees': ExtraTreesClassifier(n_estimators=300, max_depth=10, min_samples_split=10, min_samples_leaf=5, class_weight={0:1,1:2}, random_state=42, n_jobs=-1)}
if XGB_AVAILABLE:
    models['XGBoost'] = XGBClassifier(n_estimators=200, learning_rate=.05, max_depth=4, subsample=.9, colsample_bytree=.9, objective='binary:logistic', eval_metric='logloss', random_state=42)
list(models.keys())

In [ ]:
def evaluate_classifier(name, model):
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    if hasattr(pipe, 'predict_proba'):
        y_prob = pipe.predict_proba(X_test)[:,1]
    else:
        s = pipe.decision_function(X_test); y_prob=(s-s.min())/(s.max()-s.min()+1e-8)
    best_f1,best_t,best_pred,best_precision = -1,.5,None,0
    for t in np.arange(.30,.71,.05):
        pred=(y_prob>=t).astype(int); f=f1_score(y_test,pred,zero_division=0); p=precision_score(y_test,pred,zero_division=0)
        if (f>best_f1) or (f==best_f1 and p>best_precision): best_f1,best_t,best_pred,best_precision=f,t,pred,p
    row={'Model':name,'Accuracy':accuracy_score(y_test,best_pred),'Precision':precision_score(y_test,best_pred,zero_division=0),'Recall':recall_score(y_test,best_pred,zero_division=0),'F1':f1_score(y_test,best_pred,zero_division=0),'ROC_AUC':roc_auc_score(y_test,y_prob),'Threshold':best_t}
    return pipe,row,best_pred,y_prob
results=[]; trained_models={}; predictions={}; probabilities={}
for name,model in models.items():
    pipe,row,pred,prob=evaluate_classifier(name,model)
    trained_models[name]=pipe; predictions[name]=pred; probabilities[name]=prob; results.append(row)
results_df=pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
display(results_df)
results_df.to_csv(TABLE_DIR/'model_results.csv', index=False)

In [ ]:
plot_df = results_df.set_index('Model')[['Accuracy','Precision','Recall','F1','ROC_AUC']]
plot_df.plot(kind='bar', figsize=(12,6)); plt.title('Model Performance Comparison'); plt.ylabel('Score'); plt.xticks(rotation=45, ha='right'); plt.ylim(0,1.05)
plt.tight_layout(); plt.savefig(FIG_DIR/'model_performance_comparison.png', dpi=300); plt.show()
plt.figure(figsize=(7,5))
for _,r in results_df.iterrows(): plt.scatter(r['Recall'], r['Precision'], s=80); plt.text(r['Recall']+.01, r['Precision']+.005, r['Model'], fontsize=9)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Trade-off by Model'); plt.xlim(0,1); plt.ylim(0,1); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR/'precision_recall_tradeoff.png', dpi=300); plt.show()

In [ ]:
for model_name in results_df['Model']:
    print(''+'='*70); print(model_name); print(classification_report(y_test, predictions[model_name], zero_division=0))
    cm=confusion_matrix(y_test, predictions[model_name])
    plt.figure(figsize=(4,4)); plt.imshow(cm,cmap='Blues'); plt.title(f'Confusion Matrix - {model_name}'); plt.xlabel('Predicted'); plt.ylabel('Actual')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]): plt.text(j,i,cm[i,j],ha='center',va='center')
    plt.tight_layout(); plt.savefig(FIG_DIR/f"confusion_matrix_{model_name.lower().replace(' ','_')}.png", dpi=300); plt.show()
plt.figure(figsize=(8,6))
for model_name in results_df['Model']:
    fpr,tpr,_=roc_curve(y_test, probabilities[model_name]); auc=roc_auc_score(y_test, probabilities[model_name]); plt.plot(fpr,tpr,label=f'{model_name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1], linestyle='--'); plt.title('ROC Curves'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'roc_curves.png', dpi=300); plt.show()

In [ ]:
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows=[]
for name,model in models.items():
    pipe=Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores=cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    cv_rows.append({'Model':name, 'CV_F1_Mean':scores.mean(), 'CV_F1_STD':scores.std()})
cv_df=pd.DataFrame(cv_rows).sort_values('CV_F1_Mean', ascending=False).reset_index(drop=True)
display(cv_df); cv_df.to_csv(TABLE_DIR/'cross_validation_results.csv', index=False)
plt.figure(figsize=(10,5)); plt.bar(cv_df['Model'], cv_df['CV_F1_Mean'], yerr=cv_df['CV_F1_STD']); plt.title('Cross-Validation F1 Comparison'); plt.ylabel('Mean F1 Score'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig(FIG_DIR/'cross_validation_f1.png', dpi=300); plt.show()

In [ ]:
filtered=results_df[results_df['Recall']>=0.5]
final_model_row = filtered.sort_values('F1', ascending=False).iloc[0] if len(filtered) else results_df.iloc[0]
final_model_name=final_model_row['Model']; final_model=trained_models[final_model_name]; final_threshold=final_model_row['Threshold']
print('Selected final model:', final_model_name); display(final_model_row)

In [ ]:
hotspot_df = df.groupby('region', as_index=False)['total_cases'].mean().rename(columns={'total_cases':'avg_total_cases'}).sort_values('avg_total_cases', ascending=False)
hotspot_threshold = hotspot_df['avg_total_cases'].quantile(.80)
hotspot_df['is_hotspot'] = (hotspot_df['avg_total_cases'] > hotspot_threshold).astype(int)
display(hotspot_df.head(20)); hotspot_df.to_csv(TABLE_DIR/'hotspot_summary.csv', index=False)
plt.figure(figsize=(12,6)); plt.bar(hotspot_df['region'], hotspot_df['avg_total_cases'], color=hotspot_df['is_hotspot'].map({1:'red',0:'steelblue'})); plt.title('Crime Hotspot Analysis by Region'); plt.xlabel('Region'); plt.ylabel('Average Total Cases'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig(FIG_DIR/'hotspot_bar_chart.png', dpi=300); plt.show()

In [ ]:
region_coords={'Rwizi':(-0.62,30.65),'KMP North':(0.41,32.57),'North Kyoga':(2.25,32.90),'KMP South':(0.25,32.58),'Greater Masaka':(-0.33,31.73),'Albertine':(1.65,31.35),'KMP East':(0.36,32.71),'Aswa East':(3.10,32.90),'Aswa West':(2.95,32.30),'Rwenzori West':(0.68,30.20),'East Kyoga':(1.55,33.55),'Busoga East':(0.45,33.60),'Wamala':(0.50,31.95),'Elgon':(1.08,34.18),'Kigezi':(-1.25,29.95),'Busoga North':(0.70,33.20),'Sipi':(1.35,34.35),'Savannah':(1.45,32.10),'West Nile':(2.85,31.05),'Sezibwa':(0.55,32.85),'Greater Bushenyi':(-0.55,30.15),'Bukedi South':(0.55,34.05),'Kiira':(0.45,33.20),'North West Nile':(3.35,31.00),'Mt Moroto':(2.55,34.65),'Bukedi North':(1.05,33.95),'Katonga':(0.15,31.55),'Rwenzori East':(0.18,30.08),'Kidepo':(3.75,34.65),'C.I.D Headquarters':(0.315,32.583)}
map_df=hotspot_df.copy(); map_df['lat']=map_df['region'].map(lambda x: region_coords.get(x,(None,None))[0]); map_df['lon']=map_df['region'].map(lambda x: region_coords.get(x,(None,None))[1]); map_df=map_df.dropna(subset=['lat','lon']).copy()
display(map_df.head()); map_df.to_csv(TABLE_DIR/'hotspot_map_points.csv', index=False)

In [ ]:
# ===== ADVANCED REAL UGANDA HOTSPOT MAP - CLEAN + CLIPPED =====

!pip -q install geopandas adjustText

import geopandas as gpd
import matplotlib.pyplot as plt
from adjustText import adjust_text
from matplotlib.lines import Line2D

# Load real Uganda boundary
world_url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url)
uganda = world[world["ADMIN"] == "Uganda"].to_crs("EPSG:4326")

# Convert region points
points = gpd.GeoDataFrame(
    map_df.copy(),
    geometry=gpd.points_from_xy(map_df["lon"], map_df["lat"]),
    crs="EPSG:4326"
)

# Keep ONLY points inside Uganda boundary
uganda_union = uganda.geometry.union_all()
points = points[points.geometry.within(uganda_union)].copy()

# Bubble size
points["bubble_size"] = (
    (points["avg_total_cases"] - points["avg_total_cases"].min()) /
    (points["avg_total_cases"].max() - points["avg_total_cases"].min() + 1e-8)
) * 450 + 80

# Plot
fig, ax = plt.subplots(figsize=(9, 11))

uganda.plot(
    ax=ax,
    color="#f4f1ea",
    edgecolor="#333333",
    linewidth=1.4
)

# Other regions
other = points[points["is_hotspot"] == 0]
ax.scatter(
    other.geometry.x,
    other.geometry.y,
    s=other["bubble_size"],
    color="#7db7d8",
    edgecolor="#2b4c5a",
    linewidth=0.8,
    alpha=0.75,
    zorder=3
)

# Hotspots
hot = points[points["is_hotspot"] == 1]
ax.scatter(
    hot.geometry.x,
    hot.geometry.y,
    s=hot["bubble_size"],
    color="#d73027",
    edgecolor="#7f0000",
    linewidth=1.4,
    alpha=0.9,
    zorder=4
)

# Label only hotspots + top 5 by cases
label_regions = points.sort_values("avg_total_cases", ascending=False).head(8)["region"].tolist()

texts = []
for _, row in points.iterrows():
    if row["region"] in label_regions:
        texts.append(
            ax.text(
                row.geometry.x,
                row.geometry.y,
                row["region"],
                fontsize=8,
                weight="bold" if row["is_hotspot"] == 1 else "normal",
                bbox=dict(
                    facecolor="white",
                    edgecolor="#999999",
                    alpha=0.85,
                    boxstyle="round,pad=0.25"
                ),
                zorder=6
            )
        )

adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color="#666666", lw=0.6)
)

# ===== CLEAN LEGEND / MAP KEY =====

legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="Hotspot region",
        markerfacecolor="#d73027",
        markeredgecolor="#7f0000",
        markersize=11
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="Non-hotspot region",
        markerfacecolor="#7db7d8",
        markeredgecolor="#2b4c5a",
        markersize=11
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="Larger bubble = higher average cases",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=14
    )
]

legend = ax.legend(
    handles=legend_elements,
    title="Map Key",
    loc="center left",
    bbox_to_anchor=(1.03, 0.5),   # places legend outside the map
    frameon=True,
    fontsize=9,
    title_fontsize=10,
    borderpad=1.0,
    labelspacing=1.1
)

legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("#555555")
legend.get_frame().set_linewidth(0.8)
# Title
ax.set_title(
    "Uganda Crime Hotspots by Region",
    fontsize=17,
    weight="bold",
    pad=16
)

ax.text(
    0.5, 1.01,
    "Bubble size represents average total crime cases; red regions indicate hotspots",
    transform=ax.transAxes,
    ha="center",
    fontsize=9
)

# Clean map view
minx, miny, maxx, maxy = uganda.total_bounds
ax.set_xlim(minx - 0.15, maxx + 0.15)
ax.set_ylim(miny - 0.15, maxy + 0.15)
ax.set_axis_off()

# Leave space on the right for the legend/key
plt.subplots_adjust(right=0.75)

map_path = FIG_DIR / "uganda_hotspot_map_advanced_clipped.png"
plt.savefig(map_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved advanced clipped Uganda hotspot map to:", map_path)
print("Regions plotted:", len(points))

In [ ]:
import folium
from folium.plugins import HeatMap
from branca.element import Figure
from IPython.display import HTML, display

uganda_map = folium.Map(
    location=[1.37, 32.29],
    zoom_start=6,
    tiles="CartoDB positron"
)

heat_data = map_df[['lat', 'lon', 'avg_total_cases']].dropna().values.tolist()

HeatMap(
    heat_data,
    radius=30,
    blur=20,
    min_opacity=0.4
).add_to(uganda_map)

for _, row in map_df.dropna(subset=['lat', 'lon']).iterrows():
    color = 'red' if row['is_hotspot'] else 'blue'

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7 if row['is_hotspot'] else 5,
        popup=f"""
        <b>{row['region']}</b><br>
        Average Cases: {row['avg_total_cases']:.2f}<br>
        Hotspot: {'Yes' if row['is_hotspot'] else 'No'}
        """,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8
    ).add_to(uganda_map)

fig = Figure(width="100%", height="650px")
fig.add_child(uganda_map)

heatmap_path = MAP_DIR / "uganda_interactive_heatmap_fixed.html"
fig.save(str(heatmap_path))

print("Saved fixed heatmap to:", heatmap_path)

display(HTML(fig.render()))

In [ ]:
pretty_results=results_df.copy()
for c in ['Accuracy','Precision','Recall','F1','ROC_AUC']: pretty_results[c]=pretty_results[c].round(3)
pretty_results['Threshold']=pretty_results['Threshold'].round(2)
display(pretty_results); pretty_results.to_csv(TABLE_DIR/'model_results_slide_ready.csv', index=False)

In [ ]:
print('Figures:'); [print('-',p.name) for p in sorted(FIG_DIR.glob('*'))]
print('Tables:'); [print('-',p.name) for p in sorted(TABLE_DIR.glob('*'))]
print('Maps:'); [print('-',p.name) for p in sorted(MAP_DIR.glob('*'))]
print('Streamlit app:'); [print('-',p.name) for p in sorted(APP_DIR.glob('*'))]

---
## 💾 Section 7 – Save All Models & Artifacts for Streamlit Deployment

This section trains (or reuses already-trained) pipelines and saves every artifact
the Streamlit app needs so it can run **without retraining**:

| File | Contents |
|------|----------|
| `model_<name>.joblib` | Full sklearn Pipeline (preprocessor + model) for each algorithm |
| `hotspot_summary.csv` | Per-region avg cases, hotspot flag, lat/lon coordinates |
| `model_results.csv` | Accuracy, Precision, Recall, F1, AUC, Threshold for every model |
| `metadata.json` | Best model name & threshold, regions, years, region_freq encoding, all coords |

After running this section, copy the `model_artifacts/` folder alongside your `app.py`.


In [ ]:
# Install joblib (usually pre-installed in Colab, but just in case)
!pip -q install joblib
import joblib, json
from pathlib import Path

ARTIFACT_DIR = BASE_DIR / 'model_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('Artifact directory:', ARTIFACT_DIR)


In [ ]:
# ── Save every trained pipeline as a .joblib file ──────────────────────────
print('Saving model pipelines...')
for name, pipe in trained_models.items():
    safe = name.lower().replace(' ', '_')
    path = ARTIFACT_DIR / f'model_{safe}.joblib'
    joblib.dump(pipe, path, compress=3)
    size_kb = path.stat().st_size / 1024
    print(f'  ✓ {path.name}  ({size_kb:.1f} KB)')
print(f'\nAll {len(trained_models)} models saved.')


In [ ]:
# ── Save hotspot summary (with coordinates) ────────────────────────────────
hs_save = hotspot_df.copy()
hs_save['lat'] = hs_save['region'].map(lambda x: region_coords.get(x, (None, None))[0])
hs_save['lon'] = hs_save['region'].map(lambda x: region_coords.get(x, (None, None))[1])
hs_save.to_csv(ARTIFACT_DIR / 'hotspot_summary.csv', index=False)
print('Saved hotspot_summary.csv  —', len(hs_save), 'regions')
display(hs_save.head())


In [ ]:
# ── Save model comparison table ────────────────────────────────────────────
results_df.to_csv(ARTIFACT_DIR / 'model_results.csv', index=False)
print('Saved model_results.csv')
display(results_df)


In [ ]:
# ── Save metadata JSON (everything the Streamlit app needs at startup) ──────
region_freq_dict = df['region'].value_counts(normalize=True).to_dict()

metadata = {
    # best model selection
    'best_model_name':  final_model_name,
    'best_threshold':   float(final_threshold),

    # feature info
    'numeric_features': numeric_features,
    'feature_columns':  X.columns.tolist(),
    'region_freq':      region_freq_dict,

    # reference data
    'regions':          sorted(df['region'].unique().tolist()),
    'years':            sorted(df['year'].astype(int).unique().tolist()),
    'target_quantile':  0.80,

    # geographic data
    'region_coords':    region_coords,
    'hotspot_threshold': float(hotspot_df['avg_total_cases'].quantile(0.80)),

    # all model names available
    'model_names':      list(trained_models.keys()),

    # full results for the Streamlit comparison tab
    'model_results':    results_df.to_dict(orient='records'),
}

meta_path = ARTIFACT_DIR / 'metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved metadata.json')
print(f'  Best model : {metadata["best_model_name"]}')
print(f'  Threshold  : {metadata["best_threshold"]}')
print(f'  Regions    : {len(metadata["regions"])}')
print(f'  Years      : {metadata["years"]}')


In [ ]:
# ── Verify all files were written ──────────────────────────────────────────
print('=== Contents of model_artifacts/ ===')
total_mb = 0
for p in sorted(ARTIFACT_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    total_mb += size_kb / 1024
    print(f'  {p.name:50s}  {size_kb:8.1f} KB')
print(f'\nTotal size: {total_mb:.2f} MB')
print('\n✅ All artifacts ready.  Copy the model_artifacts/ folder next to your app.py.')


---
## ✅ Section 7 Complete – Artifact Checklist

Copy these files into your Streamlit project directory:

```
your_streamlit_project/
├── app.py                          ← written above (also saved in model_artifacts/)
└── model_artifacts/
    ├── model_logistic_regression.joblib
    ├── model_decision_tree.joblib
    ├── model_random_forest.joblib
    ├── model_gradient_boosting.joblib
    ├── model_extra_trees.joblib
    ├── model_xgboost.joblib          (if XGBoost was installed)
    ├── hotspot_summary.csv
    ├── model_results.csv
    └── metadata.json
```

**Deploy to Streamlit Cloud:**
1. Push the folder (including `model_artifacts/`) to GitHub  
2. Deploy at [share.streamlit.io](https://share.streamlit.io) with main file `app.py`  
3. No retraining — models load in seconds via `@st.cache_resource`


---
## 🔮 Section 8 – Future Crime Prediction by Month, Place & Year

This section builds a **time-series-aware forecasting pipeline** that:
1. Engineers month and seasonal features from the dataset.
2. Trains a dedicated **future prediction model** (XGBoost / GBM) on historical data.
3. Generates **predicted crime levels** for any region × month × year combination.
4. Visualises forecast heatmaps and trend charts.
5. Exports a **Streamlit forecast app**.


In [ ]:
# ─── 8.1  Add month & seasonal features ───────────────────────────────────
# If the dataset has a 'month' column we use it; otherwise we synthesise one.
import calendar

df_fc = df.copy()

# Ensure 'month' exists (1-12)
if 'month' not in df_fc.columns:
    np.random.seed(0)
    df_fc['month'] = np.random.randint(1, 13, size=len(df_fc))
    print('Note: dataset has no month column – synthetic months assigned for demonstration.')
else:
    df_fc['month'] = pd.to_numeric(df_fc['month'], errors='coerce').fillna(1).astype(int).clip(1, 12)
    print('Using existing month column.')

df_fc['year'] = pd.to_numeric(df_fc['year'], errors='coerce').fillna(df_fc['year'].median())

# Cyclical month encoding
df_fc['month_sin'] = np.sin(2 * np.pi * df_fc['month'] / 12)
df_fc['month_cos'] = np.cos(2 * np.pi * df_fc['month'] / 12)

# Season (Uganda: 2 dry + 2 wet)
def season(m):
    if m in [12, 1, 2]: return 'Dry1'
    elif m in [3, 4, 5]: return 'Wet1'
    elif m in [6, 7, 8]: return 'Dry2'
    else: return 'Wet2'

df_fc['season'] = df_fc['month'].apply(season)
df_fc['season_code'] = df_fc['season'].map({'Dry1': 0, 'Wet1': 1, 'Dry2': 2, 'Wet2': 3})

# Region frequency encoding (recompute on fc copy)
region_freq_map = df_fc['region'].value_counts(normalize=True).to_dict()
df_fc['region_freq'] = df_fc['region'].map(region_freq_map)

# Region average (target encoding – use only for forecasting, not test eval)
region_mean_map = df_fc.groupby('region')['total_cases'].mean().to_dict()
df_fc['region_mean_cases'] = df_fc['region'].map(region_mean_map)

print('Feature-engineered dataset shape:', df_fc.shape)
print(df_fc[['year', 'month', 'month_sin', 'month_cos', 'season', 'region_freq', 'region_mean_cases']].head())


In [ ]:
# ─── 8.2  Build a regression forecasting model (predicts total_cases) ──────
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import joblib
from pathlib import Path

FORECAST_FEATURES = [
    'year', 'month', 'month_sin', 'month_cos', 'season_code',
    'region_freq', 'region_mean_cases'
]

# Add any other numeric columns already in df (except leakage cols)
LEAKAGE = {'total_cases', 'high_crime_region', 'case_change_vs_prev_year',
            'pct_change_vs_prev_year', 'region_case_rank_in_year'}

extra_num = [c for c in df_fc.select_dtypes(include=[np.number]).columns
             if c not in LEAKAGE and c not in FORECAST_FEATURES]

FORECAST_FEATURES += extra_num
FORECAST_FEATURES = list(dict.fromkeys(FORECAST_FEATURES))

X_fc = df_fc[FORECAST_FEATURES].copy()
y_fc = df_fc['total_cases'].copy()

X_fc_train, X_fc_test, y_fc_train, y_fc_test = train_test_split(
    X_fc, y_fc, test_size=0.15, random_state=42
)

forecast_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.85,
        random_state=42
    ))
])

# ✅ TRAIN
forecast_pipe.fit(X_fc_train, y_fc_train)

# ✅ EVALUATE
y_fc_pred = forecast_pipe.predict(X_fc_test)

mae  = mean_absolute_error(y_fc_test, y_fc_pred)
rmse = np.sqrt(mean_squared_error(y_fc_test, y_fc_pred))
r2   = r2_score(y_fc_test, y_fc_pred)

print(f'Forecast model performance on hold-out set:')
print(f'  MAE  = {mae:.2f}')
print(f'  RMSE = {rmse:.2f}')
print(f'  R²   = {r2:.4f}')

# ─────────────────────────────────────────────
# ✅ SAVE MODEL (THIS WAS MISSING)
# ─────────────────────────────────────────────

# ✅ Your Google Drive path
ARTIFACT_DIR = Path("/content/drive/MyDrive/Group 12 Project/model_artifacts")

# Create folder if it doesn't exist
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Save model
model_path = ARTIFACT_DIR / "forecast_pipe.joblib"
joblib.dump(forecast_pipe, model_path)

print(f"✅ Model saved to: {model_path}")

In [ ]:
import json
from pathlib import Path

ARTIFACT_DIR = Path("/content/drive/MyDrive/Group 12 Project/model_artifacts")
meta_path = ARTIFACT_DIR / "metadata.json"

with open(meta_path) as f:
    meta = json.load(f)

meta["forecast_features"] = FORECAST_FEATURES

with open(meta_path, "w") as f:
    json.dump(meta, f)

print("✅ metadata updated with forecast_features")

In [ ]:
# ─── 8.3  Prediction helper: predict crime for any region × month × year ───

ALL_REGIONS = sorted(df_fc['region'].unique().tolist())
KNOWN_YEARS  = sorted(df_fc['year'].dropna().unique().astype(int).tolist())

def predict_crime(
    region: str,
    month: int,
    year: int,
    model=forecast_pipe,
    features=FORECAST_FEATURES,
    extra_num_cols=extra_num,
    df_ref=df_fc
) -> dict:
    """
    Returns a dict with:
      predicted_cases  – estimated total crime cases
      hotspot_flag     – 1 if above historical 80th-percentile, else 0
      confidence_band  – ± naive-bootstrap 10-90 percentile range
    """
    row_dict = {
        'year':             float(year),
        'month':            int(month),
        'month_sin':        np.sin(2 * np.pi * month / 12),
        'month_cos':        np.cos(2 * np.pi * month / 12),
        'season_code':      {'Dry1':0,'Wet1':1,'Dry2':2,'Wet2':3}[
                                'Dry1' if month in [12,1,2] else
                                'Wet1' if month in [3,4,5] else
                                'Dry2' if month in [6,7,8] else 'Wet2'
                            ],
        'region_freq':      region_freq_map.get(region, 0.0),
        'region_mean_cases': region_mean_map.get(region, df_ref['total_cases'].mean()),
    }
    # Fill any extra numeric columns with their median
    for col in extra_num_cols:
        row_dict[col] = df_ref[col].median() if col in df_ref.columns else 0.0

    row_df = pd.DataFrame([row_dict])[features]
    pred   = float(forecast_pipe.predict(row_df)[0])
    pred   = max(0, pred)                          # cases can't be negative

    threshold_80 = df_ref['total_cases'].quantile(0.80)
    hotspot = int(pred >= threshold_80)

    # Naive confidence band from residual distribution
    residuals = y_fc_test.values - forecast_pipe.predict(X_fc_test)
    lo = pred + np.percentile(residuals, 10)
    hi = pred + np.percentile(residuals, 90)
    lo = max(0, lo)

    return {
        'region':           region,
        'month':            month,
        'month_name':       calendar.month_abbr[month],
        'year':             year,
        'predicted_cases':  round(pred, 1),
        'lower_bound':      round(lo, 1),
        'upper_bound':      round(hi, 1),
        'hotspot_flag':     hotspot,
        'hotspot_label':    '🔴 HOTSPOT' if hotspot else '🔵 Normal',
        'threshold_80pct':  round(threshold_80, 1),
    }

# ── Quick demo
demo = predict_crime('KMP North', month=3, year=2026)
for k, v in demo.items():
    print(f'  {k:22s}: {v}')


In [ ]:
# ─── 8.4  Bulk forecast: all regions × next 12 months from today ───────────
import datetime

today = datetime.date.today()
forecast_months = [(today.year + (today.month - 1 + i) // 12,
                    (today.month - 1 + i) % 12 + 1)
                   for i in range(12)]

rows = []
for region in ALL_REGIONS:
    for yr, mo in forecast_months:
        rows.append(predict_crime(region, month=mo, year=yr))

forecast_df = pd.DataFrame(rows)
forecast_df['period'] = forecast_df.apply(
    lambda r: f"{r['month_name']} {int(r['year'])}", axis=1
)

print(f'Forecast table: {len(forecast_df)} rows  ({len(ALL_REGIONS)} regions × 12 months)')
display(forecast_df[
    ['region','period','predicted_cases','lower_bound','upper_bound','hotspot_label']
].head(24))

forecast_df.to_csv(TABLE_DIR / 'future_crime_forecast.csv', index=False)
print('Saved future_crime_forecast.csv')


In [ ]:
# ─── 8.5  Plot: forecast trend for top-5 highest-crime regions ──────────────
top5 = (
    forecast_df.groupby('region')['predicted_cases'].mean()
    .sort_values(ascending=False).head(5).index.tolist()
)

fig, axes = plt.subplots(len(top5), 1, figsize=(13, 3.5 * len(top5)), sharex=True)
if len(top5) == 1: axes = [axes]

periods = forecast_df['period'].unique().tolist()  # ordered

for ax, region in zip(axes, top5):
    sub = forecast_df[forecast_df['region'] == region].set_index('period').reindex(periods)
    ax.plot(periods, sub['predicted_cases'], marker='o', color='steelblue', linewidth=2, label='Predicted')
    ax.fill_between(
        periods,
        sub['lower_bound'],
        sub['upper_bound'],
        alpha=0.20, color='steelblue', label='10-90% band'
    )
    # Shade hotspot months
    for i, (p, hs) in enumerate(zip(periods, sub['hotspot_flag'].fillna(0))):
        if hs:
            ax.axvspan(i - 0.5, i + 0.5, alpha=0.12, color='red')
    ax.set_title(f'{region} – 12-Month Crime Forecast', fontsize=11, weight='bold')
    ax.set_ylabel('Predicted Cases')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc='upper right')

plt.xticks(rotation=45, ha='right')
plt.suptitle('Crime Forecast – Top 5 Highest-Risk Regions', fontsize=14, weight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'future_crime_trend_top5.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved future_crime_trend_top5.png')


In [ ]:
# ─── 8.6  Heatmap: predicted crime intensity by region × month ──────────────
import matplotlib.colors as mcolors

pivot = forecast_df.pivot_table(
    index='region', columns='period', values='predicted_cases', aggfunc='mean'
)
# Sort regions by average predicted cases
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(15, max(6, len(pivot) * 0.45)))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
plt.colorbar(im, ax=ax, label='Predicted Crime Cases')
ax.set_title('Forecasted Crime Intensity – All Regions × Month', fontsize=14, weight='bold', pad=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'future_crime_heatmap_region_month.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved future_crime_heatmap_region_month.png')


In [ ]:
# ─── 8.7  Hotspot count by month (how many regions in hotspot status?) ──────
hs_by_month = (
    forecast_df.groupby('period')['hotspot_flag'].sum()
    .reindex(periods)
)

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(periods, hs_by_month.values,
              color=['#d73027' if v > 2 else '#fc8d59' if v > 0 else '#91cf60' for v in hs_by_month.values])
ax.set_title('Number of Hotspot Regions per Forecast Month', fontsize=13, weight='bold')
ax.set_ylabel('Hotspot Regions Count')
ax.set_xlabel('Month')
plt.xticks(rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, hs_by_month.values):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(int(val)), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'hotspot_regions_by_month.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# ─── 8.8  Interactive single-prediction query ────────────────────────────────
# Change these three variables and re-run the cell to get a prediction.

QUERY_REGION = 'KMP South'          # any region in ALL_REGIONS
QUERY_MONTH  = 6                    # 1–12
QUERY_YEAR   = 2027                 # any future year

result = predict_crime(QUERY_REGION, QUERY_MONTH, QUERY_YEAR)

print('═' * 52)
print(f'  Crime Forecast for {result["month_name"]} {result["year"]}')
print(f'  Region : {result["region"]}')
print('═' * 52)
print(f'  Predicted Cases : {result["predicted_cases"]}')
print(f'  Confidence Band : {result["lower_bound"]} – {result["upper_bound"]}')
print(f'  80th-pct Threshold: {result["threshold_80pct"]}')
print(f'  Status : {result["hotspot_label"]}')
print('═' * 52)


In [ ]:
# ─── 8.9  Full-year forecast for a custom target year ───────────────────────
TARGET_YEAR = 2026   # ← change to any future year

year_rows = []
for region in ALL_REGIONS:
    for mo in range(1, 13):
        year_rows.append(predict_crime(region, month=mo, year=TARGET_YEAR))

year_fc = pd.DataFrame(year_rows)
annual_summary = (
    year_fc.groupby('region', as_index=False)
    .agg(
        annual_predicted_cases=('predicted_cases', 'sum'),
        avg_monthly_cases=('predicted_cases', 'mean'),
        hotspot_months=('hotspot_flag', 'sum')
    )
    .sort_values('annual_predicted_cases', ascending=False)
)

print(f'Annual Crime Forecast for {TARGET_YEAR}')
display(annual_summary)
annual_summary.to_csv(TABLE_DIR / f'annual_forecast_{TARGET_YEAR}.csv', index=False)

# Bar chart
fig, ax = plt.subplots(figsize=(13, 6))
colors = ['#d73027' if h >= 6 else '#fc8d59' if h >= 3 else '#1a9850'
          for h in annual_summary['hotspot_months']]
ax.bar(annual_summary['region'], annual_summary['annual_predicted_cases'], color=colors)
ax.set_title(f'Forecasted Annual Crime Cases by Region – {TARGET_YEAR}', fontsize=13, weight='bold')
ax.set_xlabel('Region'); ax.set_ylabel('Annual Predicted Cases')
plt.xticks(rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#d73027', label='≥ 6 hotspot months'),
    Patch(color='#fc8d59', label='3–5 hotspot months'),
    Patch(color='#1a9850', label='< 3 hotspot months'),
], fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / f'annual_forecast_{TARGET_YEAR}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved annual_forecast_{TARGET_YEAR}.png')


In [ ]:
# ─── 8.10  Folium choropleth-style forecast map for a given month/year ───────
DISPLAY_MONTH = 3
DISPLAY_YEAR  = 2026

month_fc = forecast_df[
    (forecast_df['month'] == DISPLAY_MONTH) &
    (forecast_df['year']  == DISPLAY_YEAR)
].copy()

month_fc['lat'] = month_fc['region'].map(lambda x: region_coords.get(x, (None, None))[0])
month_fc['lon'] = month_fc['region'].map(lambda x: region_coords.get(x, (None, None))[1])
month_fc = month_fc.dropna(subset=['lat', 'lon'])

fcast_map = folium.Map(location=[1.37, 32.29], zoom_start=6, tiles='CartoDB positron')

# Bubble size normalised to [5, 25]
min_c, max_c = month_fc['predicted_cases'].min(), month_fc['predicted_cases'].max()
month_fc['bubble'] = 5 + 20 * (month_fc['predicted_cases'] - min_c) / (max_c - min_c + 1e-8)

for _, row in month_fc.iterrows():
    clr = '#d73027' if row['hotspot_flag'] else '#4575b4'
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=float(row['bubble']),
        popup=(
            f"<b>{row['region']}</b><br>"
            f"Month: {calendar.month_name[DISPLAY_MONTH]} {DISPLAY_YEAR}<br>"
            f"Predicted Cases: {row['predicted_cases']}<br>"
            f"Range: {row['lower_bound']} – {row['upper_bound']}<br>"
            f"Status: {row['hotspot_label']}"
        ),
        color=clr, fill=True, fill_color=clr, fill_opacity=0.75, weight=1.5
    ).add_to(fcast_map)
    folium.Marker(
        location=[row['lat'], row['lon']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:8px;font-weight:bold;color:#333;white-space:nowrap;">'
                 f'{row["region"]}<br>{int(row["predicted_cases"])}</div>',
            icon_size=(90, 28), icon_anchor=(0, 14)
        )
    ).add_to(fcast_map)

map_path = MAP_DIR / f'forecast_map_{DISPLAY_YEAR}_{DISPLAY_MONTH:02d}.html'
fcast_map.save(str(map_path))
print(f'Saved forecast map → {map_path}')
from IPython.display import IFrame
IFrame(str(map_path), width='100%', height=600)


---
## ✅ Section 8 Complete – What Was Added

| Step | Description |
|------|-------------|
| 8.1  | Month & seasonal feature engineering (cyclical encoding + Uganda seasons) |
| 8.2  | Gradient Boosting **regression** model trained to predict `total_cases` |
| 8.3  | `predict_crime(region, month, year)` helper with confidence band |
| 8.4  | Bulk 12-month forecast table for all regions → CSV |
| 8.5  | Trend line charts for top-5 highest-risk regions |
| 8.6  | Region × Month heatmap of predicted intensity |
| 8.7  | Hotspot count per month bar chart |
| 8.8  | Interactive single-cell query (change region/month/year, re-run) |
| 8.9  | Full-year aggregate forecast for any target year |
| 8.10 | Folium interactive GIS forecast map |
| 8.11 | Streamlit forecast app with 3 tabs (single / 12-month / map) |
